# Этап 5. Статистическая обработка мощности

Все расчёты используют один исходный файл из репозитория и общие функции. Нет повторных диалогов загрузки архивов. `NaN` не заменяется нулём, а у каждого результата сохраняется связь с конкретным исходным файлом.

## Имена переменных

- `source` — путь к исходному файлу; `measurements` — считанная **один раз** таблица `pandas`.
- `minutes` — полная минутная сетка, включая отсутствующие строки.
- `statistics_table` — описательная статистика поминутной активной мощности по месяцам.
- `hourly_table` — средняя мощность по каждому часу суток, рассчитанная по доступным минутам.
- `month_ids` — список выбранных для профилей месяцев; `profile_table` — характеристики 1440 минут суток.
- `mean_kw` и `median_kw` — соответственно среднее арифметическое и медиана мощности, кВт.
- `std_kw` — стандартное отклонение наблюдений с `ddof=0`, кВт.
- `q1_kw`, `q3_kw` — первый и третий квартили мощности, кВт.
- `iqr_kw` — межквартильный размах `q3_kw − q1_kw`, кВт.
- `observations` — число пригодных измерений, а не число всех минут месяца.
- `coverage_record_pct` — полнота периода наблюдений, %; `coverage_calendar_pct` — полнота календарного месяца, %.

Статистика неполного месяца описывает **только известные минуты**. Усреднение не доказывает, что пропуски случайны.

In [ ]:
# Одна настройка пути к репозиторию. Исходные файлы не загружаются вручную.
from pathlib import Path
import subprocess
import sys

repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ipd_analysis.py").is_file()), None)
if repo_dir is None:
    repo_dir = Path("/content/ipd")
    if not (repo_dir / "ipd_analysis.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/sonyxoxo/ipd.git", str(repo_dir)], check=True)
sys.path.insert(0, str(repo_dir))

from ipd_analysis import choose_source, load_measurements, regular_minutes, result_folder, save_csv, save_run_metadata

# Если в data/raw несколько файлов, задайте относительный путь явно:
# source = choose_source("data/raw/household_power_consumption.zip")
source = choose_source()
measurements = load_measurements(source)  # Один pd.read_csv на весь ноутбук.
output_dir = result_folder(source)
output_dir.mkdir(parents=True, exist_ok=True)
print("Источник:", source)
print("Строк:", len(measurements))

## Описательная статистика и почасовые профили

Среднее по часу рассчитывается по **всем известным минутам соответствующего часа**, а не как простое среднее из средних при разном числе наблюдений.

- `statistics_table` — одна строка на месяц со средним, медианой, минимумом, максимумом, стандартным отклонением и квартилями.
- `hourly_table` — по одной строке на час месяца, с `n_observed` и `n_expected` для проверки полноты.

In [ ]:
from ipd_analysis import monthly_statistics, hourly_profiles, select_months, daily_profiles
from run_analysis import plot_hourly

minutes = regular_minutes(measurements)
month_ids = select_months(minutes)
statistics_table = monthly_statistics(minutes)
hourly_table = hourly_profiles(minutes, month_ids)
profile_table = daily_profiles(minutes, month_ids)
display(statistics_table.round(3))
display(hourly_table.head(8))
save_csv(statistics_table, output_dir / "05_monthly_statistics.csv")
save_csv(hourly_table, output_dir / "05_hourly_profiles.csv")
plot_hourly(hourly_table, output_dir / "05_hourly_profiles.png")
save_csv(profile_table, output_dir / "05_daily_profiles_quantiles.csv")

## Сглаживание суточного профиля

- `smoothed_table` — копия исходных средних профилей с новыми столбцами `smooth_15_kw` и `smooth_61_kw`.
- `smoothing_table` — таблица отличий между исходным и сглаженным профилями.
- `window_minutes` — ширина окна скользящего среднего: 15 или 61 минута.
- `mae_kw` — среднее абсолютное отличие двух профилей, кВт.
- `rmse_kw` — корень из среднего квадрата отличий, кВт.
- `peak_reduction_pct` — уменьшение максимума профиля после сглаживания, %.

Профиль продлевается циклически через полночь, потому что это **типичные сутки**, а не непрерывный исходный ряд. Около полуночи такая модель — допущение. Для неопределённых точек требуется не менее 80% известных соседних значений.

In [ ]:
from ipd_analysis import smooth_profiles
from run_analysis import plot_smoothing

smoothed_table, smoothing_table = smooth_profiles(profile_table)
display(smoothing_table.round(4))
save_csv(smoothed_table, output_dir / "05_smoothed_profiles.csv")
save_csv(smoothing_table, output_dir / "05_smoothing_metrics.csv")
plot_smoothing(smoothed_table, output_dir / "05_smoothing.png")

## Просмотр рисунка

- `smoothing_chart_path` — изображение результата сглаживания для всех выбранных месяцев. Оно хранится в каталоге результатов и отображается ниже.

In [ ]:
from IPython.display import Image, display

smoothing_chart_path = output_dir / "05_smoothing.png"
display(Image(filename=str(smoothing_chart_path)))

## Проверка влияния пропусков

- `experiment_profiles` — суточные профили при трёх вариантах: исходный месяц, случайные пропуски и непрерывный пропуск.
- `experiment_metrics` — сравнение сценариев, где `missing_count` — число удалённых измерений и `mae_kw` — отличие от исходного профиля.
- `incomplete_month_ids` — месяцы с отсутствующей или недопустимой мощностью.
- `complete_day_profiles` — сравнение профиля по всем известным измерениям с профилем только полных суток.
- `complete_day_metrics` — число полных суток и отличие между методами.

**Как читать рисунок:** верхняя панель показывает абсолютные суточные профили, нижняя — разность `P_с_пропусками − P_исходная` в кВт. Тонкая линия внизу — настоящие поминутные различия; толстая — только их скользящее среднее за 15 минут для наглядности. Показатель `MAE` рассчитывается по исходным различиям, без сглаживания.

Эксперимент проводится **только если есть целый календарный месяц без пропусков**. Если его нет, это честно сообщается, а не создаётся вымышленный «эталон». У новых файлов числа 43200, 8120 и 7226 не предполагаются.

Непрерывный пропуск начинается в ближайшие к середине месяца полночь; для сентября 2008 года это именно 15.09, как в исходном исследовании. `missing_start` и `missing_end` в таблице метрик фиксируют границы удалённого интервала.

In [ ]:
from ipd_analysis import monthly_coverage, missing_experiment, compare_complete_days
from run_analysis import plot_missing_experiment, plot_complete_days

experiment_profiles, experiment_metrics = missing_experiment(minutes)
if experiment_metrics.empty:
    print("Эксперимент пропущен: нет полного календарного месяца без пропусков.")
else:
    display(experiment_metrics)
    save_csv(experiment_profiles, output_dir / "05_missing_experiment_profiles.csv")
    save_csv(experiment_metrics, output_dir / "05_missing_experiment_metrics.csv")
    plot_missing_experiment(experiment_profiles, output_dir / "05_missing_experiment.png")

incomplete_month_ids = monthly_coverage(minutes).loc[
    lambda table: table["missing_or_invalid"].gt(0), "month"
].tolist()
complete_day_profiles, complete_day_metrics = compare_complete_days(minutes, incomplete_month_ids)
if not complete_day_metrics.empty:
    display(complete_day_metrics)
    save_csv(complete_day_profiles, output_dir / "05_complete_day_profiles.csv")
    save_csv(complete_day_metrics, output_dir / "05_complete_day_metrics.csv")
    plot_complete_days(complete_day_profiles, output_dir / "05_complete_day_profiles.png")

## Проверка устойчивости вывода о влиянии пропусков

- `robustness_trials` — таблица отдельных опытов: 100 повторений случайных 10%-ных пропусков и все допустимые положения непрерывного пропуска с началом в полночь.
- `robustness_summary` — средняя ошибка `mean_mae_kw`, стандартное отклонение `std_mae_kw`, минимум, максимум и квантили 2,5%/97,5% по сценариям.
- `random_repetitions` — число повторов случайного опыта; принято 100, чтобы проверить устойчивость к конкретному расположению пропусков.

Эмпирические квантили описывают **разброс результатов проведённых сценариев**, а не доверительный интервал для всех возможных домохозяйств. Если месяц без пропусков отсутствует, сравнение не проводится.

In [ ]:
from ipd_analysis import missing_experiment_robustness
from run_analysis import plot_missing_sensitivity

random_repetitions = 100
robustness_trials, robustness_summary = missing_experiment_robustness(
    minutes, random_repetitions=random_repetitions
)
if robustness_summary.empty:
    print("Невозможно оценить устойчивость: нет полного календарного месяца.")
else:
    display(robustness_summary.round(5))
    save_csv(robustness_trials, output_dir / "05_missing_robustness_trials.csv")
    save_csv(robustness_summary, output_dir / "05_missing_robustness_summary.csv")
    plot_missing_sensitivity(robustness_trials, output_dir / "05_missing_robustness.png")

## Воспроизводимость результата

Последняя ячейка сохраняет `metadata.json` с SHA-256 исходного файла, хешами исполняемого кода, версиями библиотек и датой запуска. Поэтому любой результат можно связать с конкретным источником и версией обработки.

In [ ]:
save_run_metadata(source, output_dir, {"notebook": 'статистика_мощности.ipynb'})